In [3]:
import os

def batch_rename(directory, old_extension, new_extension):
    # 獲取指定目錄中的所有檔案
    files = os.listdir(directory)

    for file in files:
        # 確保是檔案而非目錄
        if os.path.isfile(os.path.join(directory, file)):
            # 確保檔案有指定的舊副檔名
            if file.endswith(old_extension):
                # 建構新的檔案名稱，將舊副檔名替換為新副檔名
                new_name = file.replace(old_extension, new_extension)
                # 建構舊檔案和新檔案的完整路徑
                old_path = os.path.join(directory, file)
                new_path = os.path.join(directory, new_name)
                # 修改檔案名稱
                os.rename(old_path, new_path)
                print(f'Renamed: {file} to {new_name}')

# 指定目錄、舊副檔名和新副檔名
directory_path = './dataset/'
old_extension = '.jpg.chip.jpg'
new_extension = '.jpg'

# 執行批量修改副檔名的函式
batch_rename(directory_path, old_extension, new_extension)

Renamed: 100_1_0_20170110183726390.jpg.chip.jpg to 100_1_0_20170110183726390.jpg
Renamed: 100_1_2_20170105174847679.jpg.chip.jpg to 100_1_2_20170105174847679.jpg
Renamed: 101_1_2_20170105174739309.jpg.chip.jpg to 101_1_2_20170105174739309.jpg
Renamed: 10_0_0_20161220222308131.jpg.chip.jpg to 10_0_0_20161220222308131.jpg
Renamed: 10_0_0_20170103200329407.jpg.chip.jpg to 10_0_0_20170103200329407.jpg
Renamed: 10_0_0_20170103200522151.jpg.chip.jpg to 10_0_0_20170103200522151.jpg
Renamed: 10_0_0_20170103233459275.jpg.chip.jpg to 10_0_0_20170103233459275.jpg
Renamed: 10_0_0_20170104013211746.jpg.chip.jpg to 10_0_0_20170104013211746.jpg
Renamed: 10_0_0_20170110215927291.jpg.chip.jpg to 10_0_0_20170110215927291.jpg
Renamed: 10_0_0_20170110220033115.jpg.chip.jpg to 10_0_0_20170110220033115.jpg
Renamed: 10_0_0_20170110220111082.jpg.chip.jpg to 10_0_0_20170110220111082.jpg
Renamed: 10_0_0_20170110220235233.jpg.chip.jpg to 10_0_0_20170110220235233.jpg
Renamed: 10_0_0_20170110220251986.jpg.chip.jpg

In [4]:
import os
from PIL import Image, ImageDraw
from ultralytics import YOLO
import numpy as np

# 載入 yolov8 模型
yolo = YOLO('./yolov8m.pt')
# 資料夾路徑
input_folder_path = './dataset/'
output_folder_path = './pre/'


# 獲取資料夾中所有檔案
files = os.listdir(input_folder_path)

# 迴圈處理每個檔案
for filename in files:
    # 檔案完整路徑
    input_file_path = os.path.join(input_folder_path, filename)

    # 提取年齡、性別等相關信息
    parts = filename.split('_')
    age = parts[0]
    gender = parts[1]

    # 構建標註信息
    annotation = {'age': age, 'gender': gender}

    # 讀取圖片
    print(f"1-1.Opening image: {input_file_path}")
    img = Image.open(input_file_path)
    print(f"1-2.Image opened successfully.")

    # 使用 YOLO 模型進行物件檢測
    print("2-1.Predicting with YOLO model.")
    results = yolo.predict(img)
    print("2-2.Prediction completed.")

    # 保存 bbox 資訊的列表
    bbox_list = []

    # 在圖片上繪製人臉框
    draw = ImageDraw.Draw(img)

    for detection in results[0].boxes.data.numpy():
        bbox = detection[0:4].tolist()
        class_id = int(detection[5])
        x, y, w, h = bbox
        # 將相對座標轉換為相對尺寸
        x, w = x + w / 2, w
        y, h = y + h / 2, h
        # 寫入標籤信息
        label_line = f"{class_id} {x} {y} {w} {h} {age} {gender}\n"
        bbox_list.append(label_line)

        # 繪製 bbox
        draw.rectangle(bbox, outline="red", width=2)

    # 保存標註後的圖片
    output_annotated_image_path = os.path.join(output_folder_path, f"{os.path.basename(filename)}")
    print(f"3.Save img: {output_annotated_image_path}")
    img.save(output_annotated_image_path)

    # 保存標籤文件
    output_label_file_path = os.path.join(output_folder_path, f"{os.path.splitext(filename)[0]}.txt")
    with open(output_label_file_path, 'w') as f:
        for detection in results[0].boxes.data.numpy():
            bbox = detection[0:4].tolist()
            class_id = int(detection[5])
            x, y, w, h = bbox
            # 將相對座標轉換為相對尺寸
            x, w = x + w / 2, w
            y, h = y + h / 2, h
            # 寫入標籤信息
            id = str(class_id) + " " + gender + " " + age
            label_line = f"{id} {x} {y} {w} {h}\n"
            f.write(label_line)

    print("Label file saved./n-------------------------------")


1-1.Opening image: ./dataset/100_1_0_20170110183726390.jpg
1-2.Image opened successfully.
2-1.Predicting with YOLO model.



0: 640x640 1 person, 1 tie, 993.5ms
Speed: 21.0ms preprocess, 993.5ms inference, 14.6ms postprocess per image at shape (1, 3, 640, 640)



2-2.Prediction completed.
3.Save img: ./pre/100_1_0_20170110183726390.jpg
Label file saved./n-------------------------------
1-1.Opening image: ./dataset/100_1_2_20170105174847679.jpg
1-2.Image opened successfully.
2-1.Predicting with YOLO model.


0: 640x640 (no detections), 1208.1ms
Speed: 4.1ms preprocess, 1208.1ms inference, 28.6ms postprocess per image at shape (1, 3, 640, 640)



2-2.Prediction completed.
3.Save img: ./pre/100_1_2_20170105174847679.jpg
Label file saved./n-------------------------------
1-1.Opening image: ./dataset/101_1_2_20170105174739309.jpg
1-2.Image opened successfully.
2-1.Predicting with YOLO model.


...

2-2.Prediction completed.
3.Save img: ./pre/9_1_4_20170103200637399.jpg
Label file saved./n-------------------------------
1-1.Opening image: ./dataset/9_1_4_20170103200814791.jpg
1-2.Image opened successfully.
2-1.Predicting with YOLO model.


0: 640x640 2 persons, 769.7ms
Speed: 4.4ms preprocess, 769.7ms inference, 5.0ms postprocess per image at shape (1, 3, 640, 640)



2-2.Prediction completed.
3.Save img: ./pre/9_1_4_20170103200814791.jpg
Label file saved./n-------------------------------
1-1.Opening image: ./dataset/9_1_4_20170103213057382.jpg
1-2.Image opened successfully.
2-1.Predicting with YOLO model.


0: 640x640 1 person, 982.4ms
Speed: 8.4ms preprocess, 982.4ms inference, 8.8ms postprocess per image at shape (1, 3, 640, 640)


2-2.Prediction completed.
3.Save img: ./pre/9_1_4_20170103213057382.jpg
Label file saved./n-------------------------------
